# **7일차 실습: 도메인 특화 도구 만들기 및 테스트**

## 학습 목표
1. LangChain Tool 개념 이해
2. AI를 활용한 도구 자동 생성
3. 생성한 도구 테스트
4. 도메인 특화 에이전트에 적용

## 실습 단계
1. **도구 설계**: 팀 프로젝트에 필요한 도구 정의
2. **도구 생성**: AI를 활용하여 도구 코드 자동 생성
3. **도구 테스트**: 생성된 도구가 올바르게 동작하는지 확인
4. **에이전트 적용**: domain-agent에 통합 (다음 단계)

---

## 1. AI 도구 생성 프롬프트 확인

팀 프로젝트에 필요한 도구를 AI로 자동 생성하기 위한 프롬프트입니다.

**프롬프트 파일 위치**: `../../make_tool_prompt.txt`

In [1]:
# 프롬프트 파일 읽기
with open("../make_tool_prompt.txt", "r", encoding="utf-8") as f:
    prompt_template = f.read()

print("=" * 80)
print("AI 도구 생성 프롬프트")
print("=" * 80)
print(prompt_template)
print("\n✓ 프롬프트 로드 완료")
print("\n💡 이 프롬프트를 ChatGPT, Claude 등에 복사하여 사용하세요!")

AI 도구 생성 프롬프트
당신은 LangChain/LangGraph 기반 AI Agent를 위한 Python Tool 개발 전문가입니다.

아래 **[사용자 요구사항]** 에 작성된 내용을 바탕으로 LangChain Tool을 생성하세요.

---

# 사용자 요구사항

## 여기에 원하는 기능을 작성하세요.

<이곳에 원하는 Tool 기능을 작성>

---

# 구현 규칙

다음 규칙을 반드시 지키세요.

## 출력 형식

* Python 코드만 출력합니다.
* 코드 외의 설명은 출력하지 않습니다.
* `from langchain_core.tools import tool`을 사용합니다.
* `@tool(parse_docstring=True)` 데코레이터를 사용합니다.
* Python 3.11 이상 기준으로 작성합니다.

## 함수 작성 규칙

* 함수명은 기능에 맞게 작성합니다.
* 모든 매개변수에는 타입 힌트를 작성합니다.
* 반환 타입도 작성합니다.
* 필요한 import는 함수 내부에서 수행합니다.
* 가능한 표준 라이브러리를 우선 사용합니다.
* 외부 라이브러리가 필요한 경우 import도 함께 작성합니다.

## Docstring

반드시 Google Style Docstring을 작성합니다.

예시 형식

```python
"""도구 설명

Args:
    parameter1: 설명
    parameter2: 설명

Returns:
    반환값 설명
"""
```

## 예외 처리

반드시 예외 처리를 구현합니다.

형식

```python
try:
    ...
    return "성공 메시지"
except Exception as e:
    return f"실패: {str(e)}"
```

## 테스트 코드

마지막에 아래 코드를 추가합니다.

```python
print(f"도구 이름: {함수명.name}")
print(f"도구 설명: {함수명.description}")
```

## 코드 품질

* 읽기 쉬운 코드로 작성합니다.
* 

## 2. 도구 설계 가이드

### 좋은 도구의 조건

1. **단일 책임**: 하나의 명확한 기능만 수행
2. **명확한 입력/출력**: 매개변수와 반환값이 명확
3. **에러 처리**: 예외 상황을 적절히 처리
4. **좋은 설명**: Docstring으로 도구의 기능을 명확히 설명

### 도메인별 도구 예시

**쇼핑 도메인:**
- 상품 검색
- 가격 비교
- 재고 확인
- 리뷰 조회

**법령 도메인:**
- 법령 검색
- 조문 조회
- 판례 검색
- 법령 해석

**의료 도메인:**
- 증상 검색
- 병원 찾기
- 약 정보 조회
- 건강 정보 제공

**여행 도메인:**
- 항공권 검색
- 호텔 검색
- 관광지 정보
- 날씨 확인

---

## 3. 도구 생성 프로세스

### Step 1: 팀 프로젝트 도메인 및 필요한 도구 정의

**TODO: 팀에서 선택한 도메인과 필요한 도구를 작성하세요**

```
팀 도메인: [의료 정보 제공 및 안내 에이전트]

필요한 도구 목록:
1. 도구명: search_symptom_info (증상 검색 도구)
   - 입력: symptom (증상 키워드, 예: "두통과 어지러움")
   - 출력: 신뢰 가능한 의학 정보 사이트(질병관리청, 대형병원 등)에서 검색된 증상 원인·대처법·병원 방문 권고 시점 요약. 응급 키워드 감지 시 검색 없이 119/응급실 안내 즉시 반환
   - 역할: 사용자가 증상을 입력하면 관련 의학 정보를 참고용으로 제공. 진단을 내리지 않으며 항상 전문의 상담 권고 문구 포함

2. 도구명: find_hospital (병원 찾기 도구)
   - 입력: location (지역, 예: "천안시 서북구"), department (진료과, 선택 입력, 예: "내과")
   - 출력: 해당 지역/진료과 기준으로 검색된 병원 목록(명칭, 진료시간, 전화번호 등)
   - 역할: 사용자의 위치와 필요 진료과를 기반으로 방문 가능한 병원 정보를 검색·안내

3. 도구명: lookup_drug_info (약 정보 조회 도구)
   - 입력: drug_name (약품명, 예: "타이레놀")
   - 출력: 식약처 등 공식 의약품 DB에서 조회한 효능, 용법·용량, 주의사항, 부작용 정보
   - 역할: 특정 의약품에 대한 공식 정보를 제공하고, 복용 전 약사·의사 상담을 권고

4. 도구명: get_health_info (건강 정보 제공 도구)
   - 입력: topic (건강 주제, 예: "고혈압 예방법")
   - 출력: 신뢰 출처 기반 건강 관리·예방·생활 습관 정보 요약
   - 역할: 질병 예방 및 건강 관리에 관한 일반 정보를 제공하여 사용자의 건강 지식 향상을 지원
```

### Step 2: AI로 도구 생성하기

**사용 방법:**

1. search_symptom_info (증상 검색)
- 증상 키워드로 검색 (예: "두통과 어지러움")
- 신뢰 가능한 의학 정보 사이트(질병관리청, 대형병원 등)로 검색 범위 제한
- 응급 증상 키워드 감지 시 검색 없이 119/응급실 안내 즉시 반환
- 결과에 "진단 아님, 전문의 상담 권고" 문구 항상 포함

2. find_hospital (병원 찾기)
- 지역(location)과 진료과(department, 선택)로 검색
- 병원명, 진료시간, 전화번호 등 정보 반환
- 방문 전 전화 확인 권고 문구 포함

3. lookup_drug_info (약 정보 조회)
- 약품명으로 검색 (예: "타이레놀")
- 식약처 등 공식 의약품 DB로 검색 범위 제한
- 효능, 용법·용량, 주의사항, 부작용 정보 반환
- 정보 없을 시 약사/의사 상담 권고 메시지 반환

4. get_health_info (건강 정보 제공)
- 건강 주제 키워드로 검색 (예: "고혈압 예방법")
- 신뢰 출처 기반 예방/생활습관 정보 요약 반환

공통 규칙:
- LangChain @tool 데코레이터 사용
- TavilySearch를 활용하되 include_domains로 출처 제한
- 모든 도구는 docstring에 Args/Returns 명시
- 진단·처방 관련 표현 금지, 참고 정보 제공에 한정

---

## 4. 생성된 도구 코드 테스트

**TODO: AI가 생성한 도구 코드를 아래에 붙여넣으세요**

**중요:** 
- 코드를 실행하기 전에 반드시 검토하세요
- 필요한 외부 라이브러리가 있다면 먼저 설치하세요
- 실제 API 키가 필요한 경우 .env 파일에 추가하세요

In [ ]:
# 필요한 패키지 설치 및 환경 변수 로드
# (이미 설치되어 있다면 스킵됩니다)
%pip install -q langchain-core langchain-tavily python-dotenv

import os
from dotenv import load_dotenv

load_dotenv()  # 프로젝트 루트의 .env 파일에서 TAVILY_API_KEY 등을 불러옵니다

if not os.getenv("TAVILY_API_KEY"):
    print("⚠️ TAVILY_API_KEY 환경변수가 설정되어 있지 않습니다.")
    print("   .env 파일에 TAVILY_API_KEY=발급받은_키 형태로 추가해주세요.")
    print("   (키가 없어도 아래 도구 정의/구조 확인은 가능하지만, 실제 검색 호출은 실패 메시지를 반환합니다.)")
else:
    print("✓ TAVILY_API_KEY 로드 완료")


In [ ]:
from typing import Optional
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

TRUSTED_MEDICAL_DOMAINS = [
    "amc.seoul.kr",
    "snuh.org",
    "health.kdca.go.kr",
    "nedrug.mfds.go.kr",
    "mayoclinic.org",
    "webmd.com",
]

DRUG_INFO_DOMAINS = [
    "nedrug.mfds.go.kr",
    "druginfo.co.kr",
    "kimsonline.co.kr",
]

EMERGENCY_KEYWORDS = [
    "가슴 통증", "호흡 곤란", "의식 없음", "심한 출혈",
    "마비", "발작", "고열 40도", "심한 복통", "실신",
]

DISCLAIMER = (
    "\n\n⚠️ 본 정보는 참고용이며 진단·처방을 대체하지 않습니다. "
    "정확한 진단과 치료는 반드시 의료진과 상담하세요."
)

EMERGENCY_NOTICE = (
    "\n\n🚨 응급 가능성이 있는 증상입니다. "
    "즉시 119에 신고하거나 가까운 응급실을 방문하세요."
)


def _check_emergency(text: str) -> bool:
    return any(keyword in text for keyword in EMERGENCY_KEYWORDS)


@tool(parse_docstring=True)
def search_symptom_info(symptom: str) -> str:
    """증상을 기반으로 가능한 원인, 대처법, 병원 방문 권고 시점을 신뢰 가능한
    의학 정보 출처에서 검색합니다. 진단을 내리지 않습니다.

    Args:
        symptom: 검색할 증상 (예: "두통과 어지러움")

    Returns:
        증상 관련 정보 요약 및 주의사항, 또는 응급 안내 메시지
    """
    try:
        if _check_emergency(symptom):
            return f"입력하신 증상({symptom})은 응급 상황일 수 있습니다.{EMERGENCY_NOTICE}"

        search = TavilySearch(max_results=5, include_domains=TRUSTED_MEDICAL_DOMAINS)
        results = search.invoke({"query": f"{symptom} 원인 증상 대처법"})
        return f"[증상 검색 결과: {symptom}]\n{results}{DISCLAIMER}"
    except Exception as e:
        return f"실패: {str(e)}"


@tool(parse_docstring=True)
def find_hospital(location: str, department: Optional[str] = None) -> str:
    """지역과 진료과를 기반으로 병원 정보를 검색합니다.

    Args:
        location: 검색할 지역 (예: "천안시 서북구")
        department: 진료과 (예: "내과"). 없으면 종합 검색.

    Returns:
        검색된 병원 목록 및 기본 정보
    """
    try:
        dept_query = f"{department} " if department else ""
        query = f"{location} {dept_query}병원 진료시간 전화번호"

        search = TavilySearch(max_results=5)
        results = search.invoke({"query": query})
        return (
            f"[병원 검색 결과: {location} / {department or '전체'}]\n"
            f"{results}\n\n※ 방문 전 반드시 병원에 전화로 진료 가능 여부를 확인하세요."
        )
    except Exception as e:
        return f"실패: {str(e)}"


@tool(parse_docstring=True)
def lookup_drug_info(drug_name: str) -> str:
    """의약품명을 기반으로 효능, 용법·용량, 주의사항, 부작용 정보를
    공식 의약품 데이터베이스에서 조회합니다.

    Args:
        drug_name: 조회할 약품명 (예: "타이레놀")

    Returns:
        의약품 정보 요약, 또는 정보 없음 안내 메시지
    """
    try:
        search = TavilySearch(max_results=3, include_domains=DRUG_INFO_DOMAINS)
        results = search.invoke({"query": f"{drug_name} 효능 용법 주의사항 부작용"})

        if not results:
            return f"'{drug_name}'에 대한 공식 정보를 찾지 못했습니다. 약사 또는 의사와 상담하세요."

        return (
            f"[의약품 정보: {drug_name}]\n{results}\n\n"
            f"⚠️ 다른 약물과의 상호작용은 반드시 약사와 상담하세요."
        )
    except Exception as e:
        return f"실패: {str(e)}"


@tool(parse_docstring=True)
def get_health_info(topic: str) -> str:
    """건강 관리, 예방, 생활 습관 등 일반 건강 정보를 검색하여 제공합니다.

    Args:
        topic: 검색할 건강 주제 (예: "고혈압 예방법")

    Returns:
        건강 정보 요약
    """
    try:
        search = TavilySearch(max_results=5, include_domains=TRUSTED_MEDICAL_DOMAINS)
        results = search.invoke({"query": topic})
        return f"[건강 정보: {topic}]\n{results}{DISCLAIMER}"
    except Exception as e:
        return f"실패: {str(e)}"


medical_tools = [
    search_symptom_info,
    find_hospital,
    lookup_drug_info,
    get_health_info,
]



## 5. 도구 정보 확인

생성된 도구의 메타데이터를 확인합니다.

In [ ]:
tools_to_inspect = [
    search_symptom_info,
    find_hospital,
    lookup_drug_info,
    get_health_info,
]

for tool_function_name in tools_to_inspect:
    print("=" * 80)
    print("도구 정보")
    print("=" * 80)
    print(f"도구 이름: {tool_function_name.name}")
    print(f"도구 설명: {tool_function_name.description}")
    print(f"\n입력 스키마:")
    # pydantic v2에서는 .schema() 대신 .model_json_schema() 사용 (v1 호환 fallback 포함)
    schema_fn = getattr(tool_function_name.args_schema, "model_json_schema", None)
    if schema_fn is None:
        schema_fn = tool_function_name.args_schema.schema
    print(schema_fn())
    print()


## 6. 도구 단독 실행 테스트

**TODO: 다양한 입력값으로 도구를 테스트하세요**

테스트 케이스를 최소 3개 이상 작성하세요:
1. 정상 케이스
2. 엣지 케이스 (경계값)
3. 에러 케이스 (잘못된 입력)

In [ ]:
# 도구별 단독 실행 테스트 (정상 / 엣지 / 에러 케이스)

print("### 1. search_symptom_info ###")
print("테스트 1: 정상 케이스")
print(search_symptom_info.invoke({"symptom": "두통과 어지러움"}))
print()

print("테스트 2: 엣지 케이스 (응급 키워드 포함)")
print(search_symptom_info.invoke({"symptom": "가슴 통증과 호흡 곤란"}))
print()

print("테스트 3: 에러 케이스 (빈 문자열 입력)")
print(search_symptom_info.invoke({"symptom": ""}))
print()


print("### 2. find_hospital ###")
print("테스트 1: 정상 케이스")
print(find_hospital.invoke({"location": "천안시 서북구", "department": "내과"}))
print()

print("테스트 2: 엣지 케이스 (진료과 미입력)")
print(find_hospital.invoke({"location": "서울시 강남구"}))
print()

print("테스트 3: 에러 케이스 (지역 미입력)")
print(find_hospital.invoke({"location": ""}))
print()


print("### 3. lookup_drug_info ###")
print("테스트 1: 정상 케이스")
print(lookup_drug_info.invoke({"drug_name": "타이레놀"}))
print()

print("테스트 2: 엣지 케이스 (존재하지 않을 가능성이 높은 약품명)")
print(lookup_drug_info.invoke({"drug_name": "존재하지않는약품명12345"}))
print()

print("테스트 3: 에러 케이스 (빈 문자열 입력)")
print(lookup_drug_info.invoke({"drug_name": ""}))
print()


print("### 4. get_health_info ###")
print("테스트 1: 정상 케이스")
print(get_health_info.invoke({"topic": "고혈압 예방법"}))
print()

print("테스트 2: 엣지 케이스 (매우 짧은 키워드)")
print(get_health_info.invoke({"topic": "감기"}))
print()

print("테스트 3: 에러 케이스 (빈 문자열 입력)")
print(get_health_info.invoke({"topic": ""}))


## 7. 여러 도구 통합 테스트

팀에서 만든 여러 도구를 함께 테스트합니다.

**TODO: 생성한 모든 도구를 리스트로 정리하세요**

In [ ]:
# 팀에서 생성한 모든 도구를 리스트로 정리

CUSTOM_TOOLS = [
    search_symptom_info,
    find_hospital,
    lookup_drug_info,
    get_health_info,
]

print(f"총 {len(CUSTOM_TOOLS)}개의 도구가 준비되었습니다.\n")

for i, tool_item in enumerate(CUSTOM_TOOLS, 1):
    print(f"{i}. {tool_item.name}")
    print(f"   설명: {tool_item.description}")
    print()


## 프로젝트 체크리스트

**완료한 항목을 확인하세요:**

- [ ] 팀 도메인 선정 및 필요한 도구 정의 완료
- [ ] AI 프롬프트를 사용하여 도구 코드 생성 완료
- [ ] 최소 3개 이상의 도구 생성 완료
- [ ] 각 도구별 단독 실행 테스트 완료
- [ ] 정상/엣지/에러 케이스 테스트 완료
- [ ] 도구 메타데이터 확인 완료

---

## 다음 단계

생성한 도구를 domain-agent에 통합하세요:

1. `../src/domain-agent/tools.py` 파일 열기
2. TODO 주석을 참고하여 생성한 도구 코드 추가
3. `../src/domain-agent/agent.py` 파일 열기
4. TODO 주석을 참고하여 시스템 프롬프트와 도구 리스트 수정
5. LangGraph Studio로 테스트

---

## 참고 자료

- [LangChain Tools 문서](https://python.langchain.com/docs/modules/agents/tools/)
- [LangChain Custom Tools](https://python.langchain.com/docs/modules/agents/tools/custom_tools/)
- [@tool 데코레이터](https://python.langchain.com/docs/modules/agents/tools/custom_tools/#tool-decorator)